# Beats by Dre – Consumer Sentiment Analysis

**Project:** Consumer Sentiment Analysis of Beats by Dre  
**Primary product:** Beats Solo 4  
**Approach:** Data cleaning → EDA → TextBlob sentiment analysis → product comparison → actionable insights

This notebook reconstructs the analytical workflow described in the project report. The original project used Amazon review data collected through the Oxylabs API, followed by cleaning, exploratory analysis, sentiment analysis, and competitor comparison.

> **Data note:** The original raw Amazon review dataset was not included with the report. Place the original dataset in `data/raw/reviews.csv` to reproduce the analysis on the project data. A small synthetic/demo dataset is provided separately only to test that the notebook runs end-to-end; it is **not** the original project data.


## 1. Project Objectives

- Analyze consumer reviews for Beats Solo 4 and competing headphones.
- Clean and standardize review data.
- Explore rating distributions and review characteristics.
- Apply NLP-based sentiment analysis using TextBlob.
- Compare positive, neutral, and negative sentiment across products.
- Identify recurring consumer preferences and pain points.
- Translate findings into product and marketing recommendations.


## 2. Products Covered

The report describes analysis of these headphone models:

1. Beats Solo 4
2. Sony WH-CH520
3. JBL Tune 520BT
4. Sennheiser Momentum 4 Wireless
5. Marshall Monitor II ANC
6. Logitech Zone 301
7. Bose QuietComfort
8. OneOdio A70
9. Skullcandy Hesh Evo
10. Edifier WH700NB


In [ ]:
# Install dependencies if needed
# Uncomment the next line in Google Colab/Jupyter if packages are missing.
# %pip install pandas numpy matplotlib seaborn textblob scipy openpyxl python-dotenv google-generativeai

import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from textblob import TextBlob
from scipy.stats import skew, kurtosis

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "reviews.csv"

# If the notebook is opened from the notebooks/ directory, move one level up.
if not DATA_PATH.exists():
    DATA_PATH = PROJECT_ROOT.parent / "data" / "raw" / "reviews.csv"

print("Expected data path:", DATA_PATH)


## 3. Load the Review Dataset

Expected fields based on the project report include:

`review_id`, `product_name`, `product_id`, `title`, `author`, `rating`, `content`, `timestamp`, `profile_id`, `is_verified`, `helpful_count`, and `product_attributes`.

The loader below also tolerates common naming variations such as `review_text` instead of `content`.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. "
        "Add the original Amazon review dataset as data/raw/reviews.csv."
    )

df = pd.read_csv(DATA_PATH)
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
display(df.head())


### Optional Demo Dataset

If you only want to test the notebook without the original dataset, use the included synthetic file:

`data/sample/demo_reviews.csv`

Change `DATA_PATH` in the loading cell to that file. These records are synthetic and are not used to support the findings in the project report.


In [ ]:
# Standardize common column-name variations
COLUMN_ALIASES = {
    "review_text": "content",
    "review": "content",
    "text": "content",
    "product": "product_name",
    "asin": "product_id",
    "stars": "rating",
    "verified_purchase": "is_verified",
    "helpful_votes": "helpful_count",
    "date": "timestamp",
}

df = df.rename(columns={c: COLUMN_ALIASES.get(c.lower(), c) for c in df.columns})

required = ["product_name", "rating", "content"]
missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        f"Available columns: {list(df.columns)}"
    )

print("Columns after standardization:")
print(df.columns.tolist())


## 4. Data Cleaning

In [ ]:
# Basic type conversion
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df["content"] = df["content"].fillna("").astype(str)
df["product_name"] = df["product_name"].fillna("Unknown").astype(str)

# Numerical missing values: median, consistent with the project report
if df["rating"].isna().any():
    df["rating"] = df["rating"].fillna(df["rating"].median())

if "helpful_count" in df.columns:
    df["helpful_count"] = pd.to_numeric(df["helpful_count"], errors="coerce")
    df["helpful_count"] = df["helpful_count"].fillna(df["helpful_count"].median())

# Categorical missing values
for col in ["title", "author", "profile_id", "product_attributes"]:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown").astype(str)

# Timestamp parsing
if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df["review_date"] = df["timestamp"].dt.date
    df["review_year"] = df["timestamp"].dt.year
    df["review_month"] = df["timestamp"].dt.month

# Remove duplicate reviews
before = len(df)
duplicate_subset = [c for c in ["review_id", "product_id", "content"] if c in df.columns]
if duplicate_subset:
    df = df.drop_duplicates(subset=duplicate_subset)
else:
    df = df.drop_duplicates()

print(f"Removed {before - len(df):,} duplicate rows.")
print(f"Clean dataset shape: {df.shape}")


In [ ]:
# Normalize product names
PRODUCT_NORMALIZATION = {
    "Beats Solo4": "Beats Solo 4",
    "Beats Solo 4 Wireless": "Beats Solo 4",
    "Sony WHCH520": "Sony WH-CH520",
    "JBL Tune520BT": "JBL Tune 520BT",
}

df["product_name"] = df["product_name"].replace(PRODUCT_NORMALIZATION)

# Text cleaning used before NLP
def clean_text(text):
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)       # HTML
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^A-Za-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["clean_content"] = df["content"].apply(clean_text)
df["review_length"] = df["clean_content"].str.len()
df["word_count"] = df["clean_content"].str.split().str.len()

display(df[["product_name", "rating", "content", "clean_content", "word_count"]].head())


## 5. Data Quality Checks

In [ ]:
quality_report = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique()
}).sort_values("missing_pct", ascending=False)

display(quality_report)

print("Rating range:", df["rating"].min(), "to", df["rating"].max())
print("Unique products:", df["product_name"].nunique())


## 6. Exploratory Data Analysis (EDA)

In [ ]:
product_summary = (
    df.groupby("product_name")
      .agg(
          reviews=("product_name", "size"),
          mean_rating=("rating", "mean"),
          median_rating=("rating", "median"),
          rating_std=("rating", "std"),
          avg_review_length=("word_count", "mean")
      )
      .sort_values("mean_rating", ascending=False)
)

product_summary.round(3)


In [ ]:
# Rating distribution
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x="rating", bins=np.arange(0.5, 5.6, 0.5), discrete=True)
plt.title("Distribution of Review Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Reviews")
plt.tight_layout()
plt.show()


In [ ]:
# Rating distribution by product
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="rating", y="product_name")
plt.title("Rating Distribution by Product")
plt.xlabel("Rating")
plt.ylabel("Product")
plt.tight_layout()
plt.show()


In [ ]:
# Average rating and review volume
plot_df = product_summary.reset_index()

fig, ax1 = plt.subplots(figsize=(12, 6))
sns.barplot(data=plot_df, x="product_name", y="mean_rating", ax=ax1)
ax1.set_ylim(0, 5)
ax1.set_ylabel("Average Rating")
ax1.set_xlabel("")
ax1.tick_params(axis="x", rotation=60)
ax1.set_title("Average Rating and Review Volume by Product")

ax2 = ax1.twinx()
ax2.plot(range(len(plot_df)), plot_df["reviews"], marker="o")
ax2.set_ylabel("Number of Reviews")

plt.tight_layout()
plt.show()


In [ ]:
# Correlation analysis for numerical fields
numeric_cols = [c for c in ["rating", "helpful_count", "review_length", "word_count"] if c in df.columns]

if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr()
    plt.figure(figsize=(7, 5))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.show()
    display(corr)


## 7. Descriptive Statistics

The report specifically discusses mean, median, mode, variance, standard deviation, skewness, and kurtosis for ratings.


In [ ]:
rating_stats = (
    df.groupby("product_name")["rating"]
      .agg(
          mean="mean",
          median="median",
          mode=lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan,
          variance="var",
          std="std",
          skewness=skew,
          kurtosis=lambda x: kurtosis(x, fisher=False, bias=False) if len(x) > 3 else np.nan,
      )
      .sort_values("mean", ascending=False)
)

display(rating_stats.round(3))


## 8. Sentiment Analysis with TextBlob

TextBlob provides:
- **Polarity:** -1 to +1, where negative values indicate negative sentiment and positive values indicate positive sentiment.
- **Subjectivity:** 0 to 1, where higher values indicate more opinion-based language.

For this project, reviews are categorized as:
- Positive: polarity > 0
- Neutral: polarity = 0
- Negative: polarity < 0


In [ ]:
def get_sentiment(text):
    analysis = TextBlob(str(text))
    return pd.Series({
        "polarity": analysis.sentiment.polarity,
        "subjectivity": analysis.sentiment.subjectivity
    })

sentiment_scores = df["clean_content"].apply(get_sentiment)
df = pd.concat([df, sentiment_scores], axis=1)

def sentiment_label(polarity):
    if polarity > 0:
        return "Positive"
    elif polarity < 0:
        return "Negative"
    return "Neutral"

df["sentiment"] = df["polarity"].apply(sentiment_label)

display(df[["product_name", "rating", "clean_content", "polarity", "subjectivity", "sentiment"]].head())


In [ ]:
# Overall sentiment distribution
sentiment_counts = df["sentiment"].value_counts()
sentiment_pct = (sentiment_counts / len(df) * 100).round(2)

sentiment_summary = pd.DataFrame({
    "reviews": sentiment_counts,
    "percentage": sentiment_pct
})

display(sentiment_summary)

plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="sentiment", order=["Positive", "Neutral", "Negative"])
plt.title("Overall Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.tight_layout()
plt.show()


In [ ]:
# Sentiment by product
sentiment_by_product = pd.crosstab(
    df["product_name"],
    df["sentiment"],
    normalize="index"
).mul(100).round(2)

for col in ["Positive", "Neutral", "Negative"]:
    if col not in sentiment_by_product.columns:
        sentiment_by_product[col] = 0

sentiment_by_product = sentiment_by_product[["Positive", "Neutral", "Negative"]]

display(sentiment_by_product)

sentiment_by_product.plot(
    kind="bar",
    stacked=True,
    figsize=(13, 6)
)
plt.title("Sentiment Distribution by Product")
plt.xlabel("")
plt.ylabel("Percentage of Reviews")
plt.xticks(rotation=60, ha="right")
plt.legend(title="Sentiment", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
# Average polarity by product
avg_polarity = (
    df.groupby("product_name")
      .agg(
          avg_polarity=("polarity", "mean"),
          avg_subjectivity=("subjectivity", "mean"),
          reviews=("product_name", "size")
      )
      .sort_values("avg_polarity", ascending=False)
)

display(avg_polarity.round(3))


## 9. Identify Common Positive and Negative Reviews

This section surfaces reviews with the strongest positive and negative polarity so the analyst can manually inspect examples and identify recurring themes such as comfort, fit, sound quality, battery life, ANC, and app functionality.


In [ ]:
# Strongest positive reviews
positive_reviews = (
    df.sort_values("polarity", ascending=False)
      [["product_name", "rating", "polarity", "content"]]
      .head(15)
)

display(positive_reviews)

# Strongest negative reviews
negative_reviews = (
    df.sort_values("polarity", ascending=True)
      [["product_name", "rating", "polarity", "content"]]
      .head(15)
)

display(negative_reviews)


## 10. Keyword-Based Theme Exploration

The report identifies recurring themes including:
- Comfort and fit
- Sound quality and bass
- Battery life
- Active noise cancellation
- App functionality/privacy

The following lightweight approach counts reviews containing theme-related terms. This is an exploratory technique, not a supervised topic model.


In [ ]:
THEMES = {
    "Comfort & Fit": ["comfort", "comfortable", "fit", "earcup", "ears", "headband"],
    "Sound Quality": ["sound", "audio", "bass", "clarity", "music"],
    "Battery Life": ["battery", "charge", "charging", "hours"],
    "Noise Cancellation": ["noise cancellation", "anc", "noise cancelling", "isolation"],
    "App / Software": ["app", "application", "software", "privacy", "bluetooth", "connectivity"],
}

theme_rows = []

for product, group in df.groupby("product_name"):
    text = group["clean_content"].str.lower()
    total = len(group)

    for theme, keywords in THEMES.items():
        pattern = "|".join(re.escape(k) for k in keywords)
        matches = text.str.contains(pattern, regex=True, na=False)
        theme_rows.append({
            "product_name": product,
            "theme": theme,
            "review_count": int(matches.sum()),
            "percentage_of_reviews": round(matches.mean() * 100, 2)
        })

theme_df = pd.DataFrame(theme_rows)
display(theme_df.sort_values(["product_name", "review_count"], ascending=[True, False]))


## 11. Beats Solo 4 Deep Dive

In [ ]:
beats = df[df["product_name"].str.contains("Beats Solo 4", case=False, na=False)].copy()

if beats.empty:
    print("No Beats Solo 4 reviews found in the current dataset.")
else:
    print(f"Beats Solo 4 reviews: {len(beats):,}")
    print(f"Average rating: {beats['rating'].mean():.2f}")
    print(f"Average polarity: {beats['polarity'].mean():.3f}")
    display(beats["sentiment"].value_counts(normalize=True).mul(100).round(2).rename("percentage"))

    beats_theme = (
        theme_df[theme_df["product_name"].str.contains("Beats Solo 4", case=False, na=False)]
        .sort_values("review_count", ascending=False)
    )
    display(beats_theme)


## 12. Competitor Comparison

The report compares Beats Solo 4 with competitor products based on sentiment, ratings, and qualitative feedback. The code below creates a reusable comparison table directly from the review dataset.


In [ ]:
comparison = (
    df.groupby("product_name")
      .agg(
          reviews=("product_name", "size"),
          avg_rating=("rating", "mean"),
          avg_polarity=("polarity", "mean"),
          positive_pct=("sentiment", lambda x: (x == "Positive").mean() * 100),
          neutral_pct=("sentiment", lambda x: (x == "Neutral").mean() * 100),
          negative_pct=("sentiment", lambda x: (x == "Negative").mean() * 100),
      )
      .sort_values("positive_pct", ascending=False)
)

display(comparison.round(2))


## 13. Optional: Gemini AI Theme Summarization

The project report describes using Gemini AI to summarize a subset of reviews and extract deeper themes. This section is optional and requires an API key.

**Security:** Never commit an API key to GitHub. Store it in an environment variable or `.env` file that is excluded by `.gitignore`.


In [ ]:
# Optional Gemini integration
# 1. Set GEMINI_API_KEY in your environment.
# 2. Install the package if needed:
#    %pip install google-generativeai

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if GEMINI_API_KEY:
    import google.generativeai as genai

    genai.configure(api_key=GEMINI_API_KEY)

    # Model names can change over time. Replace this with a currently
    # available Gemini model in your environment.
    model = genai.GenerativeModel("gemini-1.5-flash")

    sample_reviews = (
        df[df["product_name"].str.contains("Beats Solo 4", case=False, na=False)]
        ["clean_content"]
        .dropna()
        .head(50)
        .tolist()
    )

    review_text = "\n".join(f"- {r}" for r in sample_reviews)

    prompt = f'''
    Analyze these Beats Solo 4 customer reviews.

    Identify:
    1. Common positive themes
    2. Common complaints/pain points
    3. Product features customers value
    4. Potential product improvements
    5. A concise executive summary

    Reviews:
    {review_text}
    '''

    response = model.generate_content(prompt)
    print(response.text)
else:
    print("GEMINI_API_KEY not found. Skipping optional Gemini analysis.")


## 14. Export Cleaned Data and Analysis Outputs

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
if not OUTPUT_DIR.exists():
    OUTPUT_DIR = PROJECT_ROOT.parent / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

df.to_csv(OUTPUT_DIR / "cleaned_reviews_with_sentiment.csv", index=False)
product_summary.to_csv(OUTPUT_DIR / "product_summary.csv")
rating_stats.to_csv(OUTPUT_DIR / "rating_statistics.csv")
sentiment_by_product.to_csv(OUTPUT_DIR / "sentiment_by_product.csv")
theme_df.to_csv(OUTPUT_DIR / "theme_analysis.csv")
comparison.to_csv(OUTPUT_DIR / "competitor_comparison.csv")

print("Analysis outputs saved to:", OUTPUT_DIR.resolve())


## 15. Key Business Interpretation

The original project report highlights several themes that this analytical workflow is designed to investigate:

- Beats Solo 4 was reported as having strong overall consumer satisfaction.
- Comfort, fit, sound quality, and battery life were recurring positive attributes.
- Lack of active noise cancellation was identified as a recurring pain point.
- App functionality and privacy were identified as areas of concern in some reviews.
- Bose was highlighted for noise cancellation and comfort, while Sony, Sennheiser, Marshall, and Logitech showed different competitive strengths.
- Strategic recommendations included improving comfort and fit, expanding ANC availability, improving sound customization, targeted marketing, and continuous monitoring of consumer feedback.

The exact percentages and statistics should be regenerated from the original raw dataset rather than hard-coded, because the raw Amazon review file is not included with this repository package.


## 16. Conclusion

This notebook implements an end-to-end consumer sentiment analysis workflow:

**Amazon reviews → data cleaning → EDA → descriptive statistics → TextBlob NLP → sentiment classification → theme exploration → competitor comparison → business recommendations**

This structure is suitable for a portfolio/GitHub project and can be extended with topic modeling, supervised sentiment classification, dashboarding, or automated review collection.
